<a href="https://colab.research.google.com/github/tharun8571/dl-pratice/blob/main/dlhyperparameter_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data=pd.read_csv("train.csv")

In [ ]:
data.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [ ]:
data.isnull().sum()

,0
Index,0
geohash,0
day,0
timestamp,0
demand,0
RoadType,600
NumberofLanes,0
LargeVehicles,0
Landmarks,0
Temperature,2495


In [ ]:
data.drop(['Temperature','RoadType','geohash', 'timestamp','Landmarks','Weather'],axis=1,inplace=True)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
data.head()

,Index,geohash,day,timestamp,demand,NumberofLanes,LargeVehicles,Landmarks,Weather
0,0,qp02z1,48,0:0,0.048804,1,Not Allowed,No,NaN
1,1,qp02zt,48,0:0,0.118507,3,Allowed,Yes,Sunny
2,2,qp08bj,48,0:0,0.027132,1,Not Allowed,No,Sunny
3,3,qp08gt,48,0:0,0.003272,1,Not Allowed,No,Rainy
4,4,qp02zq,48,0:0,0.010819,1,Not Allowed,No,Rainy


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
data["LargeVehicles"]=le.fit_transform(data["LargeVehicles"])


In [ ]:
x=data.iloc[:,0:6]
y=data.iloc[:,-1]

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=0)

In [ ]:
data.head()

,Index,day,demand,NumberofLanes,LargeVehicles
0,0,48,0.048804,1,1
1,1,48,0.118507,3,0
2,2,48,0.027132,1,1
3,3,48,0.003272,1,1
4,4,48,0.010819,1,1


In [ ]:
x_train = x_train.drop(['geohash', 'timestamp','Landmarks','Weather'], axis=1)
x_test = x_test.drop(['geohash', 'timestamp','Landmarks','Weather'], axis=1)

KeyError: "['geohash', 'timestamp', 'Landmarks', 'Weather'] not found in axis"

In [ ]:
pip install keras-tuner

In [ ]:
import keras_tuner as tk

In [ ]:
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential

In [ ]:
def build_model(hp):
  model=Sequential()
  model.add(Dense(6,activation="relu"))
  model.add(Dense(hp.Choice("units",min_value=6,max_value=225,step=6),activation=hp.Choice("activation",values=["relu","tanh","sigmoid"])))
  model.add(Dense(1,activation="sigmoid"))
  model.compile(optimizer=hp.Choice("optimizer",values=["adam","rmsprop","sgd"]),loss="binary_crossentropy",metrics=["accuracy"])
  return model

In [ ]:
tuner = tk.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=5)


Reloading Tuner from ./untitled_project/tuner0.json


In [ ]:
tuner.search(x_train,y_train,epochs=5,validation_data=(x_test,y_test))

In [ ]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [ ]:
def build_model(hp):
  model=Sequential()
  model.add(Dense(6,activation="relu"))
  model.add(Dense(64,activation=hp.Choice("activation",values=["relu","tanh","sigmoid"])))
  model.add(Dense(1,activation="sigmoid"))
  model.compile(optimizer=hp.Choice("optimizer",values=["adam","rmsprop","sgd"]),loss="binary_crossentropy",metrics=["accuracy"])
  return model

In [ ]:
def build_model1(hp):
  model=Sequential()
  model.add(Dense(6,activation="relu"))
  model.add(Dense(units=hp.Int("units",min_value=6,max_value=225,step=6),activation=hp.Choice("activation",values=["relu","tanh","sigmoid"])))
  model.add(Dense(1,activation="sigmoid"))
  model.compile(optimizer=hp.Choice("optimizer",values=["adam","rmsprop","sgd"]),loss="binary_crossentropy",metrics=["accuracy"])
  return model

In [ ]:
tuner=tk.RandomSearch(
    build_model1,
    objective="val_loss",
    max_trials=5
)

Reloading Tuner from ./untitled_project/tuner0.json


In [ ]:
model=tuner.search(x_train,y_train,epochs=5,validation_data=(x_test,y_test))

Trial 5 Complete [00h 00m 30s]
val_loss: 0.6464569568634033

Best val_loss So Far: 0.5843800902366638
Total elapsed time: 00h 16m 35s


In [ ]:
tuner.get_best_hyperparameters()[0].values

{'units': 84, 'activation': 'relu', 'optimizer': 'rmsprop'}

In [ ]:
model.fit(x_train,y_train,epochs=10,initial_epoch=6,validation_data=(x_test,y_test))

Epoch 7/10
1933/1933 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.6975 - loss: 0.7564 - val_accuracy: 0.7259 - val_loss: 0.5072
Epoch 8/10
1933/1933 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.6841 - loss: 0.6481 - val_accuracy: 0.7298 - val_loss: 0.5078
Epoch 9/10
1933/1933 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.6819 - loss: 0.6527 - val_accuracy: 0.6392 - val_loss: 0.6475
Epoch 10/10
1933/1933 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.7091 - loss: 0.6192 - val_accuracy: 0.3831 - val_loss: 3.4464
